<center>

# Entrega 1: Proyecto Procesamiento Alto Volumen de Datos


## **Procesamiento de Alto volumen de datos**


![Logo de la Pontificia Universidad Javeriana](Javeriana.svg)

Entregado a: Ing. John Corredor

</center>

In [1]:
## Sección 1: Importar bibliotecas generales

import os
import sys

# Bibliotecas para el procesamiento y la graficación de datos
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Importar bibliotecas especializadas 
from pylab import *

import findspark

findspark.init()

# Bibliotecas para el procesamiento de los datos usando PySpark
import pyspark.sql.functions as F

from pyspark import SparkConf, SparkContext
from pyspark.sql import SQLContext
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import *


# Bibliotecas para la creación, entrenamiento y evaluación de un modelo de machine learning from sklearn.metrics import roc_curve, auc

from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from sklearn.metrics import roc_curve, auc

# Bibliotecas de los diferentes modelos de machine learning
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.mllib.classification import SVMModel

# Sección 1: Carga de conjuntos de datos

Se cargan los sets de datos originales para su filtrado, eliminando los municipios que no son parte de este estudio, dejando únicamente Bogotá, Rioacha, Quibdó y Buenaventura. Esto permite que se pueda enfocar el resto de la investigación en el análisis de los datos y en la creación una propuesta de priorización territorial que le permita al Ministerio de Educación identificar cuáles de los municipios evaluados (Bogotá, Riohacha, Quibdó y Buenaventura) requieren intervenciones basándose en los factores que se vean que afectan la puntuación de la prueba Saber 11.

In [2]:
#Se requiere levantar la sesión para trabajar con los servicios basados en SPARK
# (Procesamiento Paralelo y Distribuido sobre Grande Volumenes de datos)

configura = SparkConf()

configura.setAppName("ProyectoProcesamiento")

sparkProyecto = SparkSession.builder.config(conf=configura).getOrCreate()

SQLContext(sparkContext=sparkProyecto.sparkContext, sparkSession=sparkProyecto)

sparkContextoProyecto = sparkProyecto.sparkContext.getOrCreate()

print("Sesion creada: ProyectoProcesamiento")

sparkProyecto

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 22:32:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Sesion creada: ProyectoProcesamiento


In [3]:
# Listado de archivos disponibles en el directorio de trabajo

!ls /Almacen/datasets/originales/

Archive.zip
DirectorioRNBP2.csv
DirectorioRNBP.csv
DirectorioRNBP.csv_bkp
IMRC-BASE-DE-DATOS-2024.csv
Internet_Fijo_Accesos_por_tecnología_y_segmento_20260525.csv
MEN_ESTADISTICAS_MATRICULA_POR_MUNICIPIOS_ES_20260525.csv
Resultados_únicos_Saber_Pro_20260525.csv
ResumenDsitribucionSGPHistorico.csv


## Carga dfBibliotecas

In [4]:
# Carga de datos desde el directorio de trabajo a un DataFrame de PySpark
dfBibliotecas = sparkProyecto.read.csv("/Almacen/datasets/originales/DirectorioRNBP2.csv", 
                                        header=True, 
                                        inferSchema=True)
dfBibliotecas.show(5)

dfBibliotecas.select("*").where(F.col("MUNICIPIO") == "BOGOTÁ, D.C.").show(10)

+------------+-----------+------------+--------------+--------------+---------------------------+-------------------+-----------------------+--------------------------+---------------------+---------+----+---------------------------+-----------------------------------+-------------------------+------------------------------------+-----------------------+-------------------+---------+--------------------+
|         CUB|CODIGO DANE|DEPARTAMENTO|     MUNICIPIO|CENTRO POBLADO|NATURALEZA DE LA BIBLIOTECA| TIPO DE BIBLIOTECA|NOMBRE DE LA BIBLIOTECA|DIRECCION DE LA BIBLIOTECA|TELEFONOS DE CONTÁCTO|EXTENSION| FAX|PÁGINA WEB DE LA BIBLIOTECA|CORREO ELECTRÓNICO DE LA BIBLIOTECA|NOMBRES DEL BIBLIOTECARIO|FECHA DE ACTUALIZACION EN EL SISTEMA|ESTADO DE LA BIBLIOTECA|            LATITUD| LONGITUD|       GEOREFERENCIA|
+------------+-----------+------------+--------------+--------------+---------------------------+-------------------+-----------------------+--------------------------+----------------

## Carga dfInternet

In [5]:
dfInternet = sparkProyecto.read.csv("/Almacen/datasets/originales/Internet_Fijo_Accesos_por_tecnología_y_segmento_20260525.csv", header=True, inferSchema=True)
dfInternet.show(5)

+----+---------+--------------------+----------------+------------+-------------+--------------------+--------------------+----------+----------------+----------------+-------------+
| AÑO|TRIMESTRE|           PROVEEDOR|COD_DEPARTAMENTO|DEPARTAMENTO|COD_MUNICIPIO|           MUNICIPIO|            SEGMENTO|TECNOLOGIA|VELOCIDAD_BAJADA|VELOCIDAD_SUBIDA|No DE ACCESOS|
+----+---------+--------------------+----------------+------------+-------------+--------------------+--------------------+----------+----------------+----------------+-------------+
|2018|        1|         EDATEL S.A.|               5|   ANTIOQUIA|         5042|SANTAFÉ DE ANTIOQUIA|         CORPORATIVO|      XDSL|            8,00|            1,00|           25|
|2019|        1|AXESS NETWORKS SO...|              52|      NARIÑO|        52473|            MOSQUERA|         CORPORATIVO| SATELITAL|            0,06|            0,06|            2|
|2019|        1|TELMEX COLOMBIA S.A.|              25|CUNDINAMARCA|        25269|    

## Carga dfIMRC

In [6]:
dfIMRC = sparkProyecto.read.csv("/Almacen/datasets/originales/IMRC-BASE-DE-DATOS-2024.csv", header=True, inferSchema=True)
dfIMRC.show(5)

26/05/25 22:32:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+------+---------------------------------------------------------------------+-----+---------------------------+------------------------------+-------------------------------+----------------------------------+---------------------------------------------+----------------------------------------------+---------------------------------------------------+------------------------------------------------------------------------------+-----------------------------------------+----------------------------+---------------------------------------------------------------+-------------------------+---------------------------------------------+---------------------+-------------------+---------------+--------------+--------------------------------------------------------+
|                  ID|            Cód_DPTO|        Departamento|        

## carga dfEducacion

In [7]:

dfEducacion = sparkProyecto.read.csv("/Almacen/datasets/originales/MEN_ESTADISTICAS_MATRICULA_POR_MUNICIPIOS_ES_20260525.csv", header=True, inferSchema=True)
dfEducacion.show(5)

+----+----------------------+-----------------------+-------------------+--------------------+-------------------+-----------+-------------+---------------+--------+---------+--------------+
| AÑO|Código delDepartamento|Nombre del Departamento|Código delMunicipio|Nombre del Municipio|TECNICA PROFESIONAL|TECNOLOGICA|UNIVERSITARIA|ESPECIALIZACION|MAESTRIA|DOCTORADO|IES CON OFERTA|
+----+----------------------+-----------------------+-------------------+--------------------+-------------------+-----------+-------------+---------------+--------+---------+--------------+
|2005|                    11|                     11|              11001|         BOGOTÁ D.C.|             63,098|     40,178|      279,544|         23,603|   7,399|      492|           106|
|2005|                    13|                     13|              13001| CARTAGENA DE INDIAS|              5,902|      4,348|       25,138|          1,260|       9|        0|            33|
|2005|                    13|                

## Carga dfSaber

In [8]:
dfSaber = sparkProyecto.read.csv("/Almacen/datasets/originales/Resultados_únicos_Saber_Pro_20260525.csv", header=True, inferSchema=True)
dfSaber.show(5)

+-------+----------------+------------------+----------------+---------------------+-----------------+---------------------+-----------------+-------------------------+---------------------------+---------------------------+--------------------+-----------------------+-----------------------+--------------------+----------------------+----------------------+-------------------+--------------------+----------------------+----------------------+-------------------+-------------------------+----------------+------------------------------+-----------------------+---------------------------+-----------------------+----------------------+-------------------------+-----------------------+------------------------+---------------------+-----------------+---------------+-----------+--------------------+------------------------+------------------------+--------------------+------------------------+----------------------+--------------------+-------------------+------------------+-----------------

## Carga dfDistribucionDinero

In [9]:
dfDistribucionDinero = sparkProyecto.read.csv("/Almacen/datasets/originales/ResumenDsitribucionSGPHistorico.csv", header=True, inferSchema=True)
dfDistribucionDinero.show(5)

+--------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------+
|            Concepto|               2026|               2025|               2024|               2023|               2022|              2021|
+--------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------+
|           Educación|$ 47209298944140.00|$ 46158590355728.00|$ 39685898906154.00|$ 30740558685361.00|$ 27883625778867.00|2.6836783602773E13|
|Prestación Servicios|$ 45614420619091.00|$ 44199520493286.00|$ 38297406750892.00|$ 29615986003258.00|$ 26756879555626.00| 2.567201609708E13|
|             Calidad| $ 1594878325049.00| $ 1959069862442.00| $ 1388492155262.00| $ 1124572682103.00| $ 1126746223241.00| 1.164767505693E12|
| Calidad (Gratuidad)|  $ 770215793265.00| $ 1188501247432.00|  $ 725347240347.00|  $ 532972334275.00|  $ 535145874114.00|  5.73167156566E11|
| Cali

# Sección 2: Limpieza de conjuntos de datos

## Limpieza dfBibliotecas
 - Se identifican los nombres de municipios que se encuentran en el dataset y se revisa si están normalizados

In [10]:
dfBibliotecas.select("Municipio").distinct().orderBy("Municipio").toPandas()["Municipio"].tolist()


[None,
 '/3223244484/',
 'ABEJORRAL',
 'ABREGO',
 'ABRIAQUÍ',
 'ACACÍAS',
 'ACANDÍ',
 'ACEVEDO',
 'ACHÍ',
 'AGRADO',
 'AGUA DE DIOS',
 'AGUACHICA',
 'AGUADA',
 'AGUADAS',
 'AGUAZUL',
 'AGUSTÍN CODAZZI',
 'AIPE',
 'ALBANIA',
 'ALBÁN',
 'ALCALÁ',
 'ALDANA',
 'ALEJANDRÍA',
 'ALGARROBO',
 'ALGECIRAS',
 'ALMAGUER',
 'ALMEIDA',
 'ALPUJARRA',
 'ALTAMIRA',
 'ALTO BAUDO',
 'ALTOS DEL ROSARIO',
 'ALVARADO',
 'AMAGÁ',
 'AMALFI',
 'AMBALEMA',
 'ANAPOIMA',
 'ANCUYA',
 'ANDALUCÍA',
 'ANDES',
 'ANGELÓPOLIS',
 'ANGOSTURA',
 'ANOLAIMA',
 'ANORÍ',
 'ANSERMA',
 'ANSERMANUEVO',
 'ANZA',
 'ANZOÁTEGUI',
 'APARTADÓ',
 'APULO',
 'APÍA',
 'AQUITANIA',
 'ARACATACA',
 'ARANZAZU',
 'ARATOCA',
 'ARAUCA',
 'ARAUQUITA',
 'ARBELÁEZ',
 'ARBOLEDA',
 'ARBOLEDAS',
 'ARBOLETES',
 'ARCABUCO',
 'ARENAL',
 'ARGELIA',
 'ARIGUANÍ',
 'ARJONA',
 'ARMENIA',
 'ARMERO',
 'ARROYOHONDO',
 'ASTREA',
 'ATACO',
 'ATRATO',
 'AYAPEL',
 'BAGADÓ',
 'BAHIA SOLANO',
 'BAJO BAUDO',
 'BALBOA',
 'BARANOA',
 'BARAYA',
 'BARBACOAS',
 'BARBOSA',
 '

Se puede ver que en este dataset se encuentran los municipios normalizados, por lo que solo se tiene que filtrar los datos por el departamento necesario

In [11]:
municipiosBiblioteca = ['BOGOTA, D.C.', 'QUIBDO', 'RIOHACHA', 'BUENAVENTURA']

dfBibliotecasFiltrado = dfBibliotecas.withColumn(
    "Municipio",
    F.when(F.col("Municipio") == "BOGOTÁ, D.C.", "BOGOTA, D.C.").otherwise(F.col("Municipio"))
)

dfBibliotecasFiltrado = dfBibliotecasFiltrado.filter(F.col("Municipio").isin(municipiosBiblioteca))
dfBibliotecasFiltrado.groupBy("Municipio").count().orderBy("Municipio").show()
dfBibliotecasFiltrado.show(5)
dfBibliotecasFiltrado.select("*").where(F.col("Municipio") == "BOGOTA, D.C.").show(5)

+------------+-----+
|   Municipio|count|
+------------+-----+
|BOGOTA, D.C.|   26|
|BUENAVENTURA|    3|
|      QUIBDO|    2|
|    RIOHACHA|    2|
+------------+-----+

+------------+-----------+------------+------------+--------------+---------------------------+------------------+-----------------------+--------------------------+---------------------+---------+----+---------------------------+-----------------------------------+-------------------------+------------------------------------+-----------------------+-------------------+--------+--------------------+
|         CUB|CODIGO DANE|DEPARTAMENTO|   Municipio|CENTRO POBLADO|NATURALEZA DE LA BIBLIOTECA|TIPO DE BIBLIOTECA|NOMBRE DE LA BIBLIOTECA|DIRECCION DE LA BIBLIOTECA|TELEFONOS DE CONTÁCTO|EXTENSION| FAX|PÁGINA WEB DE LA BIBLIOTECA|CORREO ELECTRÓNICO DE LA BIBLIOTECA|NOMBRES DEL BIBLIOTECARIO|FECHA DE ACTUALIZACION EN EL SISTEMA|ESTADO DE LA BIBLIOTECA|            LATITUD|LONGITUD|       GEOREFERENCIA|
+------------+---------

## Limpieza dfInternet

In [12]:
dfInternet.select("Municipio").distinct().orderBy("Municipio").toPandas()["Municipio"].tolist()

['ABEJORRAL',
 'ABREGO',
 'ABRIAQUÍ',
 'ACACÍAS',
 'ACANDÍ',
 'ACEVEDO',
 'ACHÍ',
 'AGRADO',
 'AGUA DE DIOS',
 'AGUACHICA',
 'AGUADA',
 'AGUADAS',
 'AGUAZUL',
 'AGUSTÍN CODAZZI',
 'AIPE',
 'ALBANIA',
 'ALBÁN',
 'ALCALÁ',
 'ALDANA',
 'ALEJANDRÍA',
 'ALGARROBO',
 'ALGECIRAS',
 'ALMAGUER',
 'ALMEIDA',
 'ALPUJARRA',
 'ALTAMIRA',
 'ALTO BAUDÓ',
 'ALTOS DEL ROSARIO',
 'ALVARADO',
 'AMAGÁ',
 'AMALFI',
 'AMBALEMA',
 'ANAPOIMA',
 'ANCUYA',
 'ANDALUCÍA',
 'ANDES',
 'ANGELÓPOLIS',
 'ANGOSTURA',
 'ANOLAIMA',
 'ANORÍ',
 'ANSERMA',
 'ANSERMANUEVO',
 'ANZA',
 'ANZOÁTEGUI',
 'APARTADÓ',
 'APULO',
 'APÍA',
 'AQUITANIA',
 'ARACATACA',
 'ARANZAZU',
 'ARATOCA',
 'ARAUCA',
 'ARAUQUITA',
 'ARBELÁEZ',
 'ARBOLEDA',
 'ARBOLEDAS',
 'ARBOLETES',
 'ARCABUCO',
 'ARENAL',
 'ARGELIA',
 'ARIGUANÍ',
 'ARJONA',
 'ARMENIA',
 'ARMERO',
 'ARROYOHONDO',
 'ASTREA',
 'ATACO',
 'ATRATO',
 'AYAPEL',
 'BAGADÓ',
 'BAHÍA SOLANO',
 'BAJO BAUDÓ',
 'BALBOA',
 'BARANOA',
 'BARAYA',
 'BARBACOAS',
 'BARBOSA',
 'BARICHARA',
 'BARRANCA D

Se puede ver los nombres del dataset de internet son iguales al dataframe anterior por lo que se procede a realizar directamente el filtrado de los datos. Sin embargo el nombre quibdó en este caso tiene tilde por lo que se procede a remover la tilde

In [13]:
# Se realiza el mismo proceso de filtrado para el DataFrame de Internet, con el fin de tener los mismos municipios en ambos DataFrames y poder realizar un análisis comparativo entre ellos.
dfInternetFiltrado = dfInternet.withColumn(
    "Municipio",
    F.when(F.col("Municipio") == "QUIBDÓ", "QUIBDO").otherwise(F.col("Municipio"))
)

dfInternetFiltrado = dfInternetFiltrado.withColumn(
    "Municipio",
    F.when(F.col("Municipio") == "BOGOTÁ, D.C.", "BOGOTA, D.C.").otherwise(F.col("Municipio"))
)

dfInternetFiltrado = dfInternetFiltrado.filter(F.col("Municipio").isin(municipiosBiblioteca))
dfInternetFiltrado.groupBy("Municipio").count().orderBy("Municipio").show()
dfInternetFiltrado.show(5)


+------------+-----+
|   Municipio|count|
+------------+-----+
|BOGOTA, D.C.|83293|
|BUENAVENTURA|10226|
|      QUIBDO| 6429|
|    RIOHACHA|12582|
+------------+-----+

+----+---------+--------------------+----------------+---------------+-------------+------------+--------------------+--------------------+----------------+----------------+-------------+
| AÑO|TRIMESTRE|           PROVEEDOR|COD_DEPARTAMENTO|   DEPARTAMENTO|COD_MUNICIPIO|   Municipio|            SEGMENTO|          TECNOLOGIA|VELOCIDAD_BAJADA|VELOCIDAD_SUBIDA|No DE ACCESOS|
+----+---------+--------------------+----------------+---------------+-------------+------------+--------------------+--------------------+----------------+----------------+-------------+
|2018|        1|UNE EPM TELECOMUN...|              11|    BOGOTÁ D.C.|        11001|BOGOTA, D.C.|         CORPORATIVO|OTRAS TECNOLOGÍAS...|           20,00|           20,00|           53|
|2019|        1|UNE EPM TELECOMUN...|              11|    BOGOTÁ D.C.|        1

## Limpieza dfIMRC

In [14]:
dfIMRC.select("Municipio").distinct().orderBy("Municipio").toPandas()["Municipio"].tolist()

['ABEJORRAL',
 'ABRIAQUÍ',
 'ACACÍAS',
 'ACANDÍ',
 'ACEVEDO',
 'ACHÍ',
 'AGRADO',
 'AGUA DE DIOS',
 'AGUACHICA',
 'AGUADA',
 'AGUADAS',
 'AGUAZUL',
 'AGUSTÍN CODAZZI',
 'AIPE',
 'ALBANIA',
 'ALBÁN',
 'ALCALÁ',
 'ALDANA',
 'ALEJANDRÍA',
 'ALGARROBO',
 'ALGECIRAS',
 'ALMAGUER',
 'ALMEIDA',
 'ALPUJARRA',
 'ALTAMIRA',
 'ALTO BAUDÓ',
 'ALTOS DEL ROSARIO',
 'ALVARADO',
 'AMAGÁ',
 'AMALFI',
 'AMBALEMA',
 'ANAPOIMA',
 'ANCUYA',
 'ANDALUCÍA',
 'ANDES',
 'ANGELÓPOLIS',
 'ANGOSTURA',
 'ANOLAIMA',
 'ANORÍ',
 'ANSERMA',
 'ANSERMANUEVO',
 'ANZOÁTEGUI',
 'ANZÁ',
 'APARTADÓ',
 'APULO',
 'APÍA',
 'AQUITANIA',
 'ARACATACA',
 'ARANZAZU',
 'ARATOCA',
 'ARAUCA',
 'ARAUQUITA',
 'ARBELÁEZ',
 'ARBOLEDA',
 'ARBOLEDAS',
 'ARBOLETES',
 'ARCABUCO',
 'ARENAL',
 'ARGELIA',
 'ARIGUANÍ',
 'ARJONA',
 'ARMENIA',
 'ARMERO',
 'ARROYOHONDO',
 'ASTREA',
 'ATACO',
 'ATRATO',
 'AYAPEL',
 'BAGADÓ',
 'BAHÍA SOLANO',
 'BAJO BAUDÓ',
 'BALBOA',
 'BARANOA',
 'BARAYA',
 'BARBACOAS',
 'BARBOSA',
 'BARICHARA',
 'BARRANCA DE UPÍA',
 '

Se puede ver que los nombres de municipios se comportan de la misma forma que los dos conjuntos de datos anteriores, por lo que se realiza la limpieza de la misma forma

In [15]:
# Se realiza el mismo proceso de filtrado para el DataFrame de Internet, con el fin de tener los mismos municipios en ambos DataFrames y poder realizar un análisis comparativo entre ellos.
dfIMRCFiltrado = dfIMRC.withColumn(
    "Municipio",
    F.when(F.col("Municipio") == "QUIBDÓ", "QUIBDO").otherwise(F.col("Municipio"))
)

dfIMRCFiltrado = dfIMRCFiltrado.withColumn(
    "Municipio",
    F.when(F.col("Municipio") == "BOGOTÁ, D.C.", "BOGOTA, D.C.").otherwise(F.col("Municipio"))
)

dfIMRCFiltrado = dfIMRCFiltrado.filter(F.col("Municipio").isin(municipiosBiblioteca))
dfIMRCFiltrado.groupBy("Municipio").count().orderBy("Municipio").show()
dfIMRCFiltrado.show(5)


+------------+-----+
|   Municipio|count|
+------------+-----+
|BOGOTA, D.C.|    1|
|BUENAVENTURA|    1|
|      QUIBDO|    1|
|    RIOHACHA|    1|
+------------+-----+

+----+--------+---------------+--------+------------+-----------------+-----------------------------+------+---------------------------------------------------------------------+-----+---------------------------+------------------------------+-------------------------------+----------------------------------+---------------------------------------------+----------------------------------------------+---------------------------------------------------+------------------------------------------------------------------------------+-----------------------------------------+----------------------------+---------------------------------------------------------------+-------------------------+---------------------------------------------+---------------------+-------------------+---------------+--------------+-----------------

## Limpieza dfEducacion

In [16]:
dfEducacion.select("Nombre del Municipio").distinct().orderBy("Nombre del Municipio").toPandas()["Nombre del Municipio"].tolist()

['ABEJORRAL',
 'ABREGO',
 'ABRIAQUI',
 'ABRIAQUÍ',
 'ACACIAS',
 'ACACÍAS',
 'ACANDI',
 'ACANDÍ',
 'ACEVEDO',
 'ACHI',
 'ACHÍ',
 'AGRADO',
 'AGUA DE DIOS',
 'AGUACHICA',
 'AGUADA',
 'AGUADAS',
 'AGUAZUL',
 'AGUSTIN CODAZZI',
 'AGUSTÍN CODAZZI',
 'AIPE',
 'ALBAN',
 'ALBAN (SAN JOSE)',
 'ALBANIA',
 'ALBANIA (GUAJIRA)',
 'ALBÁN',
 'ALCALA',
 'ALCALÁ',
 'ALDANA',
 'ALEJANDRIA',
 'ALEJANDRÍA',
 'ALGARROBO',
 'ALGECIRAS',
 'ALMAGUER',
 'ALMEIDA',
 'ALPUJARRA',
 'ALTAMIRA',
 'ALTO BAUDO',
 'ALTO BAUDÓ',
 'ALTOS DEL ROSARIO',
 'ALVARADO',
 'AMAGA',
 'AMAGÁ',
 'AMALFI',
 'AMBALEMA',
 'ANAPOIMA',
 'ANCUYA',
 'ANDALUCIA',
 'ANDALUCÍA',
 'ANDES',
 'ANGELOPOLIS',
 'ANGELÓPOLIS',
 'ANGOSTURA',
 'ANOLAIMA',
 'ANORI',
 'ANORÍ',
 'ANSERMA',
 'ANSERMANUEVO',
 'ANZA',
 'ANZOATEGUI',
 'ANZOÁTEGUI',
 'ANZÁ',
 'APARTADO',
 'APARTADÓ',
 'APIA',
 'APULO',
 'APÍA',
 'AQUITANIA',
 'ARACATACA',
 'ARANZAZU',
 'ARATOCA',
 'ARAUCA',
 'ARAUQUITA',
 'ARBELAEZ',
 'ARBELÁEZ',
 'ARBOLEDA',
 'ARBOLEDAS',
 'ARBOLETES',
 'A

En este caso, el conjunto de datos contiene varias maneras en las que se puede tomar el nombre de bogotá ( 'BOGOTA D.C.', 'BOGOTÁ D.C.', 'BOGOTÁ, D.C.',), por lo que se opta por cambiar todas estás ocurrencias por "BOGOTA, D.C."

In [17]:
# Se realiza el mismo proceso de filtrado para el DataFrame de educación, con el fin de tener los mismos municipios en ambos DataFrames y poder realizar un análisis comparativo entre ellos.
dfEducacionFiltrado = dfEducacion.withColumn(
    "Nombre del Municipio",
    F.when(F.col("Nombre del Municipio") == "QUIBDÓ", "QUIBDO").otherwise(F.col("Nombre del Municipio"))
)

dfEducacionFiltrado = dfEducacionFiltrado.withColumn(
    "Nombre del Municipio",
    F.when(F.col("Nombre del Municipio") == "BOGOTÁ, D.C.", "BOGOTA, D.C.").otherwise(F.col("Nombre del Municipio"))
)

dfEducacionFiltrado = dfEducacionFiltrado.withColumn(
    "Nombre del Municipio",
    F.when(F.col("Nombre del Municipio") == "BOGOTA D.C.", "BOGOTA, D.C.").otherwise(F.col("Nombre del Municipio"))
)

dfEducacionFiltrado = dfEducacionFiltrado.filter(F.col("Nombre del Municipio").isin(municipiosBiblioteca))
dfEducacionFiltrado.groupBy("Nombre del Municipio").count().orderBy("Nombre del Municipio").show()
dfEducacionFiltrado.show(5)


+--------------------+-----+
|Nombre del Municipio|count|
+--------------------+-----+
|        BOGOTA, D.C.|   14|
|        BUENAVENTURA|   31|
|              QUIBDO|   49|
|            RIOHACHA|   49|
+--------------------+-----+

+----+----------------------+-----------------------+-------------------+--------------------+-------------------+-----------+-------------+---------------+--------+---------+--------------+
| AÑO|Código delDepartamento|Nombre del Departamento|Código delMunicipio|Nombre del Municipio|TECNICA PROFESIONAL|TECNOLOGICA|UNIVERSITARIA|ESPECIALIZACION|MAESTRIA|DOCTORADO|IES CON OFERTA|
+----+----------------------+-----------------------+-------------------+--------------------+-------------------+-----------+-------------+---------------+--------+---------+--------------+
|2005|                    27|                     27|              27001|              QUIBDO|                494|         81|            0|              0|       0|        0|             1|
|20

## Limpieza dfSaber

In [18]:
dfSaber.select("ESTU_MCPIO_RESIDE").distinct().orderBy("ESTU_MCPIO_RESIDE").toPandas()["ESTU_MCPIO_RESIDE"].tolist()

[None,
 'ABEJORRAL',
 'ABRIAQUÍ',
 'ACACÍAS',
 'ACANDÍ',
 'ACEVEDO',
 'ACHÍ',
 'AGRADO',
 'AGUA DE DIOS',
 'AGUACHICA',
 'AGUADA',
 'AGUADAS',
 'AGUAZUL',
 'AGUSTÍN CODAZZI',
 'AIPE',
 'ALBANIA',
 'ALBÁN',
 'ALCALÁ',
 'ALDANA',
 'ALEJANDRÍA',
 'ALGARROBO',
 'ALGECIRAS',
 'ALMAGUER',
 'ALMEIDA',
 'ALPUJARRA',
 'ALTAMIRA',
 'ALTO BAUDÓ',
 'ALTOS DEL ROSARIO',
 'ALVARADO',
 'AMAGÁ',
 'AMALFI',
 'AMBALEMA',
 'ANAPOIMA',
 'ANCUYÁ',
 'ANDALUCÍA',
 'ANDES',
 'ANGELÓPOLIS',
 'ANGOSTURA',
 'ANOLAIMA',
 'ANORÍ',
 'ANSERMA',
 'ANSERMANUEVO',
 'ANZOÁTEGUI',
 'ANZÁ',
 'APARTADÓ',
 'APULO',
 'APÍA',
 'AQUITANIA',
 'ARACATACA',
 'ARANZAZU',
 'ARATOCA',
 'ARAUCA',
 'ARAUQUITA',
 'ARBELÁEZ',
 'ARBOLEDA',
 'ARBOLEDAS',
 'ARBOLETES',
 'ARCABUCO',
 'ARENAL',
 'ARGELIA',
 'ARIGUANÍ',
 'ARJONA',
 'ARMENIA',
 'ARMERO',
 'ARROYOHONDO',
 'ASTREA',
 'ATACO',
 'ATLANTA',
 'ATRATO',
 'AYAPEL',
 'BAGADÓ',
 'BAHÍA SOLANO',
 'BAJO BAUDÓ',
 'BALBOA',
 'BARANOA',
 'BARAYA',
 'BARBACOAS',
 'BARBOSA',
 'BARCELONA',
 'BA

In [19]:
# Se realiza el mismo proceso de filtrado para el DataFrame de la prueba saber, con el fin de tener los mismos municipios en ambos DataFrames y poder realizar un análisis comparativo entre ellos.
dfSaberFiltrado = dfSaber.withColumn(
    "ESTU_MCPIO_RESIDE",
    F.when(F.col("ESTU_MCPIO_RESIDE") == "QUIBDÓ", "QUIBDO").otherwise(F.col("ESTU_MCPIO_RESIDE"))
)

dfSaberFiltrado = dfSaberFiltrado.withColumn(
    "ESTU_MCPIO_RESIDE",
    F.when(F.col("ESTU_MCPIO_RESIDE") == "BOGOTÁ D.C.", "BOGOTA, D.C.").otherwise(F.col("ESTU_MCPIO_RESIDE"))
)

dfSaberFiltrado = dfSaberFiltrado.filter(F.col("ESTU_MCPIO_RESIDE").isin(municipiosBiblioteca))
dfSaberFiltrado.groupBy("ESTU_MCPIO_RESIDE").count().orderBy("ESTU_MCPIO_RESIDE").show()
dfSaberFiltrado.show(5)


+-----------------+------+
|ESTU_MCPIO_RESIDE| count|
+-----------------+------+
|     BOGOTA, D.C.|326784|
|     BUENAVENTURA|  4309|
|           QUIBDO|  7145|
|         RIOHACHA|  5532|
+-----------------+------+

+-------+----------------+------------------+----------------+---------------------+-----------------+---------------------+-----------------+-------------------------+---------------------------+---------------------------+--------------------+-----------------------+-----------------------+--------------------+----------------------+----------------------+-------------------+--------------------+----------------------+----------------------+-------------------+-------------------------+----------------+------------------------------+-----------------------+---------------------------+-----------------------+----------------------+-------------------------+-----------------------+------------------------+---------------------+-----------------+---------------+-----------+

## Limpieza dfDistribucionDinero

Este conjunto de datos, al ser únicamente datos sobre la cantidad de dinero destinado a diferentes sectores publicos, no cuenta con diferentes municipios, por lo que no es necesario su limpieza

In [20]:
dfDistribucionDinero.show()

+--------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------+
|            Concepto|               2026|               2025|               2024|               2023|               2022|              2021|
+--------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------+
|           Educación|$ 47209298944140.00|$ 46158590355728.00|$ 39685898906154.00|$ 30740558685361.00|$ 27883625778867.00|2.6836783602773E13|
|Prestación Servicios|$ 45614420619091.00|$ 44199520493286.00|$ 38297406750892.00|$ 29615986003258.00|$ 26756879555626.00| 2.567201609708E13|
|             Calidad| $ 1594878325049.00| $ 1959069862442.00| $ 1388492155262.00| $ 1124572682103.00| $ 1126746223241.00| 1.164767505693E12|
| Calidad (Gratuidad)|  $ 770215793265.00| $ 1188501247432.00|  $ 725347240347.00|  $ 532972334275.00|  $ 535145874114.00|  5.73167156566E11|
| Cali

# Guardado de datos filtrados
Se guardan los conjuntos de datos filtrados para su posterior análisis y modelado.


In [21]:
dfBibliotecasFiltrado.toPandas().to_csv("/Almacen/datasets/filtrados/bibliotecas_filtrado_municipios.csv", index=False)
dfInternetFiltrado.toPandas().to_csv("/Almacen/datasets/filtrados/internet_filtrado_municipios.csv", index=False)
dfIMRCFiltrado.toPandas().to_csv("/Almacen/datasets/filtrados/imrc_filtrado_municipios.csv", index=False)
dfEducacionFiltrado.toPandas().to_csv("/Almacen/datasets/filtrados/nivel_educativo_filtrado_municipios.csv", index=False)
dfSaberFiltrado.toPandas().to_csv("/Almacen/datasets/filtrados/icfes_filtrado_municipios.csv", index=False)
dfDistribucionDinero.toPandas().to_csv("/Almacen/datasets/filtrados/ResumenDsitribucionSGPHistoricoEducacion.csv", index=False)